# 05 Ensemble Selection

Ce notebook choisit les modèles réellement utilisés en production par l'API.

Il répond à trois objectifs du projet :

1. transformer les benchmarks des notebooks 03 et 04 en une décision finale explicable ;
2. vérifier si le vote doux apporte un gain mesurable face au meilleur modèle seul ;
3. produire `models/ensemble_config.json`, la configuration lue par `src/api/model_loader.py`.

L'idée reste volontairement simple : les CSV locaux sont la source de vérité, la sélection finale garde les trois meilleurs modèles avec une diversité raisonnable, puis le gain du vote doux est mesuré après rechargement des checkpoints.


## Choix méthodologiques

- Les fichiers CSV locaux dans `models/` sont la source principale de vérité. MLflow/DagsHub reste utile pour l'audit, mais le notebook doit rester exécutable même sans connexion.
- Les modèles de screening ne sont pas utilisés pour la configuration finale. On attend les modèles fine-tunés, plus représentatifs de la performance finale.
- Le score de sélection combine plusieurs signaux : F1 macro, balanced accuracy, log loss, temps d'inférence et signes d'overfit.
- Les métriques OOD ne sont utilisées pour sélectionner que si la couverture PlantDoc est fiable pour la tâche. Elles sont toujours affichées comme diagnostic.
- La diversité d'architecture est souhaitable mais pas forcée aveuglément : on limite à deux modèles par famille, sauf si les résultats disponibles ne permettent pas mieux.
- L'architecture finale reste homogène : chaque tâche utilise trois modèles sélectionnés et un vote doux. Le gain face au meilleur modèle seul est mesuré et discuté dans le rapport, mais il ne remplace pas automatiquement l'ensemble.
- Le notebook est prévu pour être lancé après la fin du notebook 04. Il affiche l'état d'avancement, puis s'arrête clairement si une tâche n'a pas assez de modèles fine-tunés.


In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.models.build import densenet_preprocess_input, resnet_v2_preprocess_input

REPO_ROOT

2026-04-17 19:03:16.413436: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 19:03:16.587195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776445396.653130    5361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776445396.672426    5361 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776445396.829058    5361 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

PosixPath('/home/thomashebert99/code/thomashebert99/Plant-disease-detection')

## Configuration

Les chemins suivent exactement la structure générée par les notebooks précédents. `EXPECTED_TASKS` sert aussi de checklist : si une tâche n'a pas encore de `finetune_results.csv`, elle restera marquée comme incomplète.

In [2]:
SEED = 42
IMAGE_SIZE = 224
EVAL_BATCH_SIZE = 4
PREFETCH_BUFFER_SIZE = 1
ENSEMBLE_SIZE = 3
REQUIRE_COMPLETE_RESULTS = True
MAX_MODELS_PER_FAMILY = 2
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

DATA_ROOT = REPO_ROOT / "data" / "processed"
OOD_ROOT = REPO_ROOT / "data" / "test_ood"
MODEL_ROOT = REPO_ROOT / "models"
DISEASE_MODEL_ROOT = MODEL_ROOT / "diseases"
ENSEMBLE_OUTPUT_DIR = MODEL_ROOT / "ensemble"
EVALUATION_OUTPUT_DIR = ENSEMBLE_OUTPUT_DIR / "evaluations"
ENSEMBLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPECIES_CONFIGS = {
    "tomato": {"display_name": "Tomate", "expected_classes": 10},
    "apple": {"display_name": "Pommier", "expected_classes": 4},
    "grape": {"display_name": "Vigne", "expected_classes": 4},
    "corn": {"display_name": "Mais", "expected_classes": 4},
    "potato": {"display_name": "Pomme de terre", "expected_classes": 3},
    "pepper": {"display_name": "Poivron", "expected_classes": 2},
    "strawberry": {"display_name": "Fraisier", "expected_classes": 2},
}

EXPECTED_TASKS = ["species", *SPECIES_CONFIGS.keys()]

FINAL_RANKING_METRICS_ID = [
    ("eval_test_f1_macro", 0.30, True),
    ("eval_test_balanced_accuracy", 0.25, True),
    ("eval_val_f1_macro", 0.15, True),
    ("eval_test_log_loss", 0.10, False),
    ("eval_test_ms_per_image", 0.05, False),
    ("loss_gap", 0.05, False),
    ("overfit_gap", 0.05, False),
]

FINAL_RANKING_METRICS_OOD_RELIABLE = [
    ("eval_ood_f1_macro", 0.15, True),
    ("eval_ood_balanced_accuracy", 0.10, True),
]

PREPROCESSOR_CUSTOM_OBJECTS = {
    "DenseNet121": {
        "preprocess_input": tf.keras.applications.densenet.preprocess_input,
        "densenet_preprocess_input": densenet_preprocess_input,
        "plant_disease>densenet_preprocess_input": densenet_preprocess_input,
    },
    "DenseNet169": {
        "preprocess_input": tf.keras.applications.densenet.preprocess_input,
        "densenet_preprocess_input": densenet_preprocess_input,
        "plant_disease>densenet_preprocess_input": densenet_preprocess_input,
    },
    "ResNet50V2": {
        "preprocess_input": tf.keras.applications.resnet_v2.preprocess_input,
        "resnet_v2_preprocess_input": resnet_v2_preprocess_input,
        "plant_disease>resnet_v2_preprocess_input": resnet_v2_preprocess_input,
    },
    "ResNet101V2": {
        "preprocess_input": tf.keras.applications.resnet_v2.preprocess_input,
        "resnet_v2_preprocess_input": resnet_v2_preprocess_input,
        "plant_disease>resnet_v2_preprocess_input": resnet_v2_preprocess_input,
    },
}

np.random.seed(SEED)
tf.random.set_seed(SEED)

## Charger les résultats des notebooks 03 et 04

Chaque ligne correspond à un modèle entraîné. On garde les checkpoints locaux, les métriques et les informations utiles pour reconstruire l'ordre des classes.

In [3]:
def checkpoint_exists(value: object) -> bool:
    if pd.isna(value):
        return False
    path = Path(str(value))
    if path.exists():
        return True
    if not path.is_absolute() and (REPO_ROOT / path).exists():
        return True
    return False


def resolve_checkpoint_path(value: object) -> Path:
    path = Path(str(value))
    if path.exists():
        return path
    if not path.is_absolute():
        candidate = REPO_ROOT / path
        if candidate.exists():
            return candidate
    return path


def class_names_from_directory(directory: Path) -> list[str]:
    if not directory.exists():
        return []
    return sorted(path.name for path in directory.iterdir() if path.is_dir())


def parse_class_names(value: object, fallback: list[str]) -> list[str]:
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list) and parsed:
                return [str(item) for item in parsed]
        except json.JSONDecodeError:
            pass
    return fallback


def load_task_results(task: str) -> pd.DataFrame:
    if task == "species":
        results_path = MODEL_ROOT / "species" / "finetune_results.csv"
        task_type = "species"
        display_name = "Espèce"
        fallback_class_names = class_names_from_directory(DATA_ROOT / "species" / "train")
    else:
        results_path = DISEASE_MODEL_ROOT / task / "finetune_results.csv"
        task_type = "disease"
        display_name = SPECIES_CONFIGS[task]["display_name"]
        fallback_class_names = class_names_from_directory(DATA_ROOT / task / "train")

    if not results_path.exists():
        return pd.DataFrame()

    frame = pd.read_csv(results_path)
    if frame.empty:
        return frame

    frame = frame.copy()
    frame["task"] = task
    frame["task_type"] = task_type
    frame["display_name"] = frame.get("display_name", display_name)
    frame["results_path"] = str(results_path)
    frame["checkpoint_exists"] = frame["checkpoint_path"].map(checkpoint_exists)
    frame["class_names"] = frame.get("class_names", pd.Series([None] * len(frame))).map(
        lambda value: json.dumps(parse_class_names(value, fallback_class_names))
    )
    frame["num_classes"] = frame["class_names"].map(lambda value: len(json.loads(value)))
    if "ood_reliable" not in frame.columns:
        frame["ood_reliable"] = False
    return frame


def load_all_results() -> pd.DataFrame:
    frames = [load_task_results(task) for task in EXPECTED_TASKS]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


all_results = load_all_results()
summary_columns = [
    "task", "run_name", "architecture", "architecture_family", "stage",
    "checkpoint_exists", "eval_val_f1_macro", "eval_test_f1_macro",
    "eval_ood_f1_macro", "ood_reliable", "checkpoint_path",
]
display(all_results[[column for column in summary_columns if column in all_results.columns]])

,task,run_name,architecture,architecture_family,stage,checkpoint_exists,eval_val_f1_macro,eval_test_f1_macro,eval_ood_f1_macro,ood_reliable,checkpoint_path
0,species,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,fine_tuning,True,0.999716,0.999769,0.605973,False,/home/thomashebert99/code/thomashebert99/Plant...
1,species,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,ConvNeXt,fine_tuning,True,0.999738,0.999563,0.704791,False,/home/thomashebert99/code/thomashebert99/Plant...
2,species,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,fine_tuning,True,0.999699,0.999416,0.651511,False,/home/thomashebert99/code/thomashebert99/Plant...
3,species,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,fine_tuning,True,0.999825,0.999084,0.521123,False,/home/thomashebert99/code/thomashebert99/Plant...
4,tomato,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,ConvNeXt,fine_tuning,True,0.993009,0.989785,0.175945,False,/home/thomashebert99/code/thomashebert99/Plant...
5,tomato,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,fine_tuning,True,0.986137,0.988261,0.137239,False,/home/thomashebert99/code/thomashebert99/Plant...
6,tomato,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,fine_tuning,True,0.982956,0.984912,0.156663,False,/home/thomashebert99/code/thomashebert99/Plant...
7,tomato,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,MobileNet,fine_tuning,True,0.973908,0.972126,0.122917,False,/home/thomashebert99/code/thomashebert99/Plant...
8,apple,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,fine_tuning,True,1.000000,1.000000,0.331464,False,/home/thomashebert99/code/thomashebert99/Plant...
9,apple,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,fine_tuning,True,0.999337,0.999338,0.297298,False,/home/thomashebert99/code/thomashebert99/Plant...


## État d'avancement des tâches

Cette cellule sert de tableau de bord. Comme ce notebook doit être lancé après le notebook 04, chaque tâche doit idéalement avoir trois checkpoints fine-tunés utilisables.

In [4]:
def readiness_report(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task in EXPECTED_TASKS:
        task_results = results[results["task"] == task] if not results.empty else pd.DataFrame()
        usable = task_results[task_results.get("checkpoint_exists", False) == True] if not task_results.empty else task_results
        rows.append(
            {
                "task": task,
                "available_finetuned_runs": int(len(task_results)),
                "usable_checkpoints": int(len(usable)),
                "ready_for_ensemble": bool(len(usable) >= ENSEMBLE_SIZE),
                "ready_for_single_model_check": bool(len(usable) >= 1),
            }
        )
    return pd.DataFrame(rows)


readiness = readiness_report(all_results)
display(readiness)

if REQUIRE_COMPLETE_RESULTS:
    incomplete_tasks = readiness.loc[~readiness["ready_for_ensemble"], "task"].tolist()
    if incomplete_tasks:
        raise RuntimeError(
            "Résultats incomplets pour le notebook 05. "
            f"Tâches à terminer dans le notebook 04 ou 03: {incomplete_tasks}"
        )

,task,available_finetuned_runs,usable_checkpoints,ready_for_ensemble,ready_for_single_model_check
0,species,4,4,True,True
1,tomato,4,4,True,True
2,apple,4,4,True,True
3,grape,4,4,True,True
4,corn,4,4,True,True
5,potato,4,4,True,True
6,pepper,4,4,True,True
7,strawberry,4,4,True,True


## Score de sélection

Le score est un rang pondéré. Il évite de choisir un modèle uniquement parce qu'il a gagné d'un cheveu sur la validation, tout en pénalisant les modèles lents, instables ou trop overfités.

Pour l'OOD, on applique une règle prudente : si PlantDoc ne couvre pas correctement les classes d'une tâche, les métriques OOD restent dans le tableau mais ne pèsent pas dans le score final.

In [5]:
def bool_series(values: pd.Series) -> pd.Series:
    return values.astype(str).str.lower().isin(["true", "1", "yes"])


def ranking_metric_specs(task_results: pd.DataFrame) -> list[tuple[str, float, bool]]:
    metric_specs = list(FINAL_RANKING_METRICS_ID)
    if "ood_reliable" in task_results.columns and bool_series(task_results["ood_reliable"]).all():
        metric_specs.extend(FINAL_RANKING_METRICS_OOD_RELIABLE)
    return metric_specs


def add_ranking_score(
    summary: pd.DataFrame,
    metric_specs: list[tuple[str, float, bool]],
) -> pd.DataFrame:
    if summary.empty:
        return summary

    ranked = summary.copy()
    score_columns = []
    for metric, weight, higher_is_better in metric_specs:
        if metric not in ranked.columns:
            continue
        values = pd.to_numeric(ranked[metric], errors="coerce")
        if values.notna().sum() == 0:
            continue
        fill_value = values.min() if higher_is_better else values.max()
        score_column = f"score_{metric}"
        ranked[score_column] = values.fillna(fill_value).rank(
            method="average",
            pct=True,
            ascending=higher_is_better,
        ) * weight
        score_columns.append(score_column)

    if not score_columns:
        ranked["ranking_score"] = 0.0
    else:
        ranked["ranking_score"] = ranked[score_columns].sum(axis=1)

    secondary = "eval_test_f1_macro" if "eval_test_f1_macro" in ranked.columns else "eval_val_f1_macro"
    ranked = ranked.sort_values(["ranking_score", secondary], ascending=[False, False])
    ranked["ranking_position"] = np.arange(1, len(ranked) + 1)
    return ranked


def rank_task_results(task_results: pd.DataFrame) -> pd.DataFrame:
    usable = task_results[task_results["checkpoint_exists"]].copy()
    if usable.empty:
        return usable
    return add_ranking_score(usable, ranking_metric_specs(usable))


ranked_results = []
for task, task_results in all_results.groupby("task") if not all_results.empty else []:
    ranked_results.append(rank_task_results(task_results))
ranked_results = pd.concat(ranked_results, ignore_index=True) if ranked_results else pd.DataFrame()

ranking_columns = [
    "task", "ranking_position", "run_name", "architecture", "architecture_family",
    "ranking_score", "eval_test_f1_macro", "eval_test_balanced_accuracy",
    "eval_ood_f1_macro", "eval_test_ms_per_image", "overfit_gap", "ood_reliable",
]
display(ranked_results[[column for column in ranking_columns if column in ranked_results.columns]])

,task,ranking_position,run_name,architecture,architecture_family,ranking_score,eval_test_f1_macro,eval_test_balanced_accuracy,eval_ood_f1_macro,eval_test_ms_per_image,overfit_gap,ood_reliable
0,apple,1,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.93750,1.000000,1.000000,0.331464,5.651444,-0.002059,False
1,apple,2,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,0.63750,0.999338,0.999337,0.297298,6.118357,0.000049,False
2,apple,3,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,MobileNet,0.46250,0.998671,0.998675,0.311666,5.340088,-0.001078,False
3,apple,4,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,0.33750,0.994597,0.994709,0.360005,6.084959,-0.001814,False
4,corn,1,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,0.88750,0.991666,0.991183,0.222895,8.383349,0.007020,False
5,corn,2,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,0.68750,0.990920,0.990705,0.227760,7.947105,0.006550,False
6,corn,3,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.48750,0.990207,0.990005,0.248979,7.821624,0.008682,False
7,corn,4,DenseNet121_ft_l50_lr1e_05,DenseNet121,DenseNet,0.31250,0.981037,0.979931,0.224441,10.034012,-0.000743,False
8,grape,1,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,0.82500,1.000000,1.000000,0.314364,8.971754,-0.001636,False
9,grape,2,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.70625,0.999300,0.999306,0.329085,8.857575,-0.001002,False


## Comparer les stratégies de sélection

On compare trois options avant de choisir :

- `top3_brut` : les trois meilleurs modèles selon le score ;
- `top3_max2_family` : les trois meilleurs avec au maximum deux modèles par famille ;
- `one_per_family` : une diversité stricte, avec au maximum un modèle par famille.

Pour ce projet, on retient `top3_max2_family`. C'est un compromis simple : il évite les ensembles trop homogènes sans remplacer un très bon modèle par un modèle nettement moins bon uniquement pour varier l'architecture.


In [6]:
def select_top_models(ranked: pd.DataFrame, *, top_k: int = ENSEMBLE_SIZE) -> pd.DataFrame:
    if ranked.empty:
        return ranked
    selected = ranked.head(top_k).copy()
    selected["selected_rank"] = np.arange(1, len(selected) + 1)
    return selected


def select_diverse_models(
    ranked: pd.DataFrame,
    *,
    top_k: int = ENSEMBLE_SIZE,
    max_per_family: int = MAX_MODELS_PER_FAMILY,
) -> pd.DataFrame:
    if ranked.empty:
        return ranked

    selected_indices = []
    family_counts: dict[str, int] = {}
    for row_index, row in ranked.iterrows():
        family = row.get("architecture_family", "unknown")
        if family_counts.get(family, 0) >= max_per_family:
            continue
        selected_indices.append(row_index)
        family_counts[family] = family_counts.get(family, 0) + 1
        if len(selected_indices) == top_k:
            break

    for row_index, _ in ranked.iterrows():
        if len(selected_indices) == top_k:
            break
        if row_index not in selected_indices:
            selected_indices.append(row_index)

    selected = ranked.loc[selected_indices].copy()
    selected["selected_rank"] = np.arange(1, len(selected) + 1)
    return selected


def summarize_selection_strategy(task: str, strategy_name: str, selected: pd.DataFrame) -> dict[str, object]:
    return {
        "task": task,
        "strategy": strategy_name,
        "model_count": int(len(selected)),
        "families": " + ".join(selected.get("architecture_family", pd.Series(dtype=str)).astype(str).tolist()),
        "architectures": " + ".join(selected.get("architecture", pd.Series(dtype=str)).astype(str).tolist()),
        "mean_test_f1_macro": float(selected["eval_test_f1_macro"].mean()) if "eval_test_f1_macro" in selected else np.nan,
        "min_test_f1_macro": float(selected["eval_test_f1_macro"].min()) if "eval_test_f1_macro" in selected else np.nan,
        "mean_test_log_loss": float(selected["eval_test_log_loss"].mean()) if "eval_test_log_loss" in selected else np.nan,
        "mean_test_ms_per_image": float(selected["eval_test_ms_per_image"].mean()) if "eval_test_ms_per_image" in selected else np.nan,
        "mean_ood_f1_macro": float(selected["eval_ood_f1_macro"].mean()) if "eval_ood_f1_macro" in selected else np.nan,
    }


strategy_rows = []
selected_frames = []

for task in EXPECTED_TASKS:
    task_ranked = ranked_results[ranked_results["task"] == task] if not ranked_results.empty else pd.DataFrame()
    top3_selection = select_top_models(task_ranked)
    max2_selection = select_diverse_models(task_ranked, max_per_family=2)
    strict_family_selection = select_diverse_models(task_ranked, max_per_family=1)

    strategy_rows.extend(
        [
            summarize_selection_strategy(task, "top3_brut", top3_selection),
            summarize_selection_strategy(task, "top3_max2_family", max2_selection),
            summarize_selection_strategy(task, "one_per_family", strict_family_selection),
        ]
    )
    selected_frames.append(max2_selection)

strategy_comparison = pd.DataFrame(strategy_rows)
strategy_comparison_path = ENSEMBLE_OUTPUT_DIR / "selection_strategy_comparison.csv"
strategy_comparison.to_csv(strategy_comparison_path, index=False)

selected_models = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame()
selection_path = ENSEMBLE_OUTPUT_DIR / "selection_summary.csv"
selected_models.to_csv(selection_path, index=False)

selection_columns = [
    "task", "selected_rank", "run_name", "architecture", "architecture_family",
    "ranking_score", "eval_test_f1_macro", "eval_ood_f1_macro", "checkpoint_path",
]
display(strategy_comparison)
display(selected_models[[column for column in selection_columns if column in selected_models.columns]])
print(f"Comparaison des stratégies sauvegardée dans: {strategy_comparison_path}")
print(f"Sélection candidate sauvegardée dans: {selection_path}")


,task,strategy,model_count,families,architectures,mean_test_f1_macro,min_test_f1_macro,mean_test_log_loss,mean_test_ms_per_image,mean_ood_f1_macro
0,species,top3_brut,3,EfficientNet + ConvNeXt + EfficientNet,EfficientNetB0 + ConvNeXtTiny + EfficientNetB1,0.999583,0.999416,0.001907,7.169792,0.654092
1,species,top3_max2_family,3,EfficientNet + ConvNeXt + EfficientNet,EfficientNetB0 + ConvNeXtTiny + EfficientNetB1,0.999583,0.999416,0.001907,7.169792,0.654092
2,species,one_per_family,3,EfficientNet + ConvNeXt + MobileNet,EfficientNetB0 + ConvNeXtTiny + MobileNetV3Large,0.999472,0.999084,0.001927,7.061903,0.610629
3,tomato,top3_brut,3,ConvNeXt + MobileNet + EfficientNet,ConvNeXtTiny + MobileNetV3Large + EfficientNetB0,0.987652,0.984912,0.040046,8.290607,0.156616
4,tomato,top3_max2_family,3,ConvNeXt + MobileNet + EfficientNet,ConvNeXtTiny + MobileNetV3Large + EfficientNetB0,0.987652,0.984912,0.040046,8.290607,0.156616
5,tomato,one_per_family,3,ConvNeXt + MobileNet + EfficientNet,ConvNeXtTiny + MobileNetV3Large + EfficientNetB0,0.987652,0.984912,0.040046,8.290607,0.156616
6,apple,top3_brut,3,EfficientNet + MobileNet + MobileNet,EfficientNetB0 + MobileNetV3Large + MobileNetV...,0.999336,0.998671,0.003649,5.703296,0.313476
7,apple,top3_max2_family,3,EfficientNet + MobileNet + MobileNet,EfficientNetB0 + MobileNetV3Large + MobileNetV...,0.999336,0.998671,0.003649,5.703296,0.313476
8,apple,one_per_family,3,EfficientNet + MobileNet + MobileNet,EfficientNetB0 + MobileNetV3Large + MobileNetV...,0.999336,0.998671,0.003649,5.703296,0.313476
9,grape,top3_brut,3,EfficientNet + EfficientNet + ConvNeXt,EfficientNetB1 + EfficientNetB0 + ConvNeXtTiny,0.999277,0.998532,0.002393,9.820958,0.319800


,task,selected_rank,run_name,architecture,architecture_family,ranking_score,eval_test_f1_macro,eval_ood_f1_macro,checkpoint_path
0,species,1,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.85000,0.999769,0.605973,/home/thomashebert99/code/thomashebert99/Plant...
1,species,2,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,ConvNeXt,0.63750,0.999563,0.704791,/home/thomashebert99/code/thomashebert99/Plant...
2,species,3,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,0.46250,0.999416,0.651511,/home/thomashebert99/code/thomashebert99/Plant...
3,tomato,1,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,ConvNeXt,0.83750,0.989785,0.175945,/home/thomashebert99/code/thomashebert99/Plant...
4,tomato,2,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,0.68750,0.988261,0.137239,/home/thomashebert99/code/thomashebert99/Plant...
5,tomato,3,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.50000,0.984912,0.156663,/home/thomashebert99/code/thomashebert99/Plant...
6,apple,1,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.93750,1.000000,0.331464,/home/thomashebert99/code/thomashebert99/Plant...
7,apple,2,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,0.63750,0.999338,0.297298,/home/thomashebert99/code/thomashebert99/Plant...
8,apple,3,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,MobileNet,0.46250,0.998671,0.311666,/home/thomashebert99/code/thomashebert99/Plant...
9,grape,1,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,0.82500,1.000000,0.314364,/home/thomashebert99/code/thomashebert99/Plant...


Comparaison des stratégies sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/selection_strategy_comparison.csv
Sélection candidate sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/selection_summary.csv


## Fonctions d'évaluation du vote doux

Le vote doux consiste à moyenner les probabilités de chaque modèle :

`proba_ensemble = moyenne(proba_modele_1, proba_modele_2, proba_modele_3)`

On charge les modèles un par un pour éviter de garder trop de checkpoints en mémoire.

In [7]:
def image_path_records(root: Path, class_names: list[str]) -> pd.DataFrame:
    records = []
    for class_index, class_name in enumerate(class_names):
        class_dir = root / class_name
        if not class_dir.exists():
            continue
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(
                    {
                        "image_path": str(image_path),
                        "class_index": class_index,
                        "class_name": class_name,
                    }
                )
    return pd.DataFrame(records)


def load_raw_image(image_path: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image, label


def build_path_dataset(root: Path, class_names: list[str]) -> tuple[tf.data.Dataset | None, pd.DataFrame]:
    records = image_path_records(root, class_names)
    if records.empty:
        return None, records
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            records["image_path"].to_numpy(),
            records["class_index"].to_numpy(dtype="int32"),
        )
    )
    dataset = dataset.map(load_raw_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(EVAL_BATCH_SIZE).prefetch(PREFETCH_BUFFER_SIZE)
    return dataset, records


def task_dataset_roots(task: str) -> dict[str, Path]:
    if task == "species":
        return {
            "test": DATA_ROOT / "species" / "test",
            "ood": OOD_ROOT,
        }
    return {
        "test": DATA_ROOT / task / "test",
        "ood": OOD_ROOT / task,
    }


def build_eval_datasets(task: str, class_names: list[str]) -> dict[str, tuple[tf.data.Dataset | None, pd.DataFrame]]:
    roots = task_dataset_roots(task)
    return {name: build_path_dataset(root, class_names) for name, root in roots.items()}


def load_checkpoint_model(architecture: str, checkpoint_path: Path) -> tf.keras.Model:
    return tf.keras.models.load_model(
        str(checkpoint_path),
        compile=False,
        safe_mode=False,
        custom_objects=PREPROCESSOR_CUSTOM_OBJECTS.get(architecture, {}),
    )


def predict_dataset(model: tf.keras.Model, dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray, float]:
    y_true = []
    y_score_batches = []
    image_count = 0
    started_at = time.perf_counter()
    for images, labels in dataset:
        predictions = model.predict(images, verbose=0)
        y_score_batches.append(predictions)
        y_true.extend(labels.numpy().astype(int).tolist())
        image_count += int(images.shape[0])
    elapsed = time.perf_counter() - started_at
    return np.array(y_true, dtype="int32"), np.concatenate(y_score_batches, axis=0), elapsed


def classification_metrics(
    y_true: np.ndarray,
    y_score: np.ndarray,
    *,
    num_classes: int,
    elapsed_seconds: float,
) -> dict[str, float]:
    y_pred = np.argmax(y_score, axis=1)
    labels = np.arange(num_classes)
    image_count = max(1, len(y_true))
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "log_loss": float(log_loss(y_true, y_score, labels=labels)),
        "ms_per_image": float((elapsed_seconds / image_count) * 1000),
        "image_count": float(image_count),
    }
    top_k = min(2, num_classes - 1) if num_classes > 2 else 1
    if top_k > 1:
        metrics[f"top{top_k}_accuracy"] = float(top_k_accuracy_score(y_true, y_score, k=top_k, labels=labels))
    return metrics


def predict_checkpoint(row: pd.Series, dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray, float]:
    tf.keras.backend.clear_session()
    model = None
    try:
        model = load_checkpoint_model(row["architecture"], resolve_checkpoint_path(row["checkpoint_path"]))
        return predict_dataset(model, dataset)
    finally:
        if model is not None:
            del model
        tf.keras.backend.clear_session()
        gc.collect()


def evaluate_single_and_ensemble(
    task: str,
    selected: pd.DataFrame,
    dataset_name: str,
    dataset: tf.data.Dataset,
    num_classes: int,
) -> list[dict[str, object]]:
    rows = []
    ensemble_score_sum = None
    ensemble_elapsed = 0.0
    reference_y_true = None

    for _, model_row in selected.iterrows():
        y_true, y_score, elapsed = predict_checkpoint(model_row, dataset)
        reference_y_true = y_true if reference_y_true is None else reference_y_true
        if not np.array_equal(reference_y_true, y_true):
            raise ValueError(f"Ordre des labels incohérent pendant l'évaluation de {task}/{dataset_name}.")

        single_metrics = classification_metrics(
            y_true,
            y_score,
            num_classes=num_classes,
            elapsed_seconds=elapsed,
        )
        rows.append(
            {
                "task": task,
                "dataset": dataset_name,
                "model_type": "single",
                "run_name": model_row["run_name"],
                "architecture": model_row["architecture"],
                **single_metrics,
            }
        )

        ensemble_score_sum = y_score if ensemble_score_sum is None else ensemble_score_sum + y_score
        ensemble_elapsed += elapsed

    if ensemble_score_sum is not None and len(selected) >= 2:
        ensemble_score = ensemble_score_sum / len(selected)
        ensemble_metrics = classification_metrics(
            reference_y_true,
            ensemble_score,
            num_classes=num_classes,
            elapsed_seconds=ensemble_elapsed,
        )
        rows.append(
            {
                "task": task,
                "dataset": dataset_name,
                "model_type": "ensemble_soft_vote",
                "run_name": "+".join(selected["run_name"].tolist()),
                "architecture": "+".join(selected["architecture"].tolist()),
                **ensemble_metrics,
            }
        )
    return rows

## Évaluer le gain de l'ensemble, tâche par tâche

L'évaluation recharge plusieurs checkpoints Keras et peut être coûteuse en mémoire. Pour éviter de faire planter WSL, on ne lance plus toutes les tâches dans une seule cellule.

Principe :

1. exécuter la cellule de fonctions ci-dessous ;
2. exécuter une seule cellule de tâche à la fois ;
3. attendre la fin et vérifier le CSV sauvegardé ;
4. passer à la tâche suivante.

Chaque tâche écrit immédiatement son résultat dans `models/ensemble/evaluations/<task>_ensemble_evaluation.csv`. La synthèse globale recharge ensuite ces fichiers.


In [8]:
def task_evaluation_path(task: str) -> Path:
    return EVALUATION_OUTPUT_DIR / f"{task}_ensemble_evaluation.csv"


def load_saved_evaluations() -> pd.DataFrame:
    frames = []
    for evaluation_file in sorted(EVALUATION_OUTPUT_DIR.glob("*_ensemble_evaluation.csv")):
        frame = pd.read_csv(evaluation_file)
        frame["evaluation_file"] = str(evaluation_file)
        frames.append(frame)

    evaluation = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not evaluation.empty:
        evaluation_path = ENSEMBLE_OUTPUT_DIR / "ensemble_evaluation.csv"
        evaluation.to_csv(evaluation_path, index=False)
    return evaluation


def evaluate_task_ensemble(task: str, datasets_to_run: tuple[str, ...] = ("test", "ood")) -> pd.DataFrame:
    task_selection = selected_models[selected_models["task"] == task] if not selected_models.empty else pd.DataFrame()
    if len(task_selection) < ENSEMBLE_SIZE:
        raise RuntimeError(
            f"{task}: {len(task_selection)} modèle(s) sélectionné(s), "
            f"{ENSEMBLE_SIZE} requis pour le vote doux."
        )

    class_names = json.loads(task_selection.iloc[0]["class_names"])
    datasets = build_eval_datasets(task, class_names)
    rows = []
    output_path = task_evaluation_path(task)

    print(f"{task}: {len(task_selection)} modèle(s), {len(class_names)} classe(s)")
    print(f"Sauvegarde: {output_path}")

    for dataset_name in datasets_to_run:
        dataset, records = datasets.get(dataset_name, (None, pd.DataFrame()))
        if dataset is None or records.empty:
            print(f"  - {dataset_name}: aucune image disponible")
            continue

        print(f"  - {dataset_name}: {len(records)} images")
        rows.extend(
            evaluate_single_and_ensemble(
                task=task,
                selected=task_selection,
                dataset_name=dataset_name,
                dataset=dataset,
                num_classes=len(class_names),
            )
        )

        partial_evaluation = pd.DataFrame(rows)
        partial_evaluation.to_csv(output_path, index=False)
        print(f"    sauvegarde intermédiaire: {len(partial_evaluation)} ligne(s)")

        del dataset
        gc.collect()
        tf.keras.backend.clear_session()

    task_evaluation = pd.DataFrame(rows)
    if not task_evaluation.empty:
        task_evaluation.to_csv(output_path, index=False)
        display(task_evaluation)
        print(f"Évaluation {task} sauvegardée dans: {output_path}")
    else:
        print(f"{task}: aucune évaluation produite.")

    global evaluation
    evaluation = load_saved_evaluations()
    return task_evaluation


evaluation = load_saved_evaluations()
if evaluation.empty:
    print("Aucune évaluation sauvegardée pour le moment. Lance les cellules par tâche ci-dessous.")
else:
    display(evaluation)
    print(f"Évaluations rechargées depuis: {EVALUATION_OUTPUT_DIR}")


Aucune évaluation sauvegardée pour le moment. Lance les cellules par tâche ci-dessous.


### Évaluation espèce

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [9]:
evaluate_task_ensemble("species")


I0000 00:00:1776445400.733959    5361 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21452 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


species: 3 modèle(s), 7 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/species_ensemble_evaluation.csv
  - test: 10092 images


I0000 00:00:1776445403.539332    5432 service.cc:152] XLA service 0x7ed0480478a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776445403.539356    5432 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-04-17 19:03:23.619571: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776445404.034337    5432 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-04-17 19:03:24.418545: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2031_0', 176 bytes spill stores, 524 bytes spill loads

2026-04-17 19:03:24.468802: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2

    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 2322 images


2026-04-17 19:09:44.488530: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2031_0', 176 bytes spill stores, 524 bytes spill loads

2026-04-17 19:09:44.499201: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2031', 104 bytes spill stores, 104 bytes spill loads

2026-04-17 19:09:44.748750: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2031', 12 bytes spill stores, 12 bytes spill loads

2026-04-17 19:09:44.765167: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2031', 4460 bytes spill stores, 4504 bytes spill loads

2026-04-17 19:09:44.867891: I 

    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,species,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.999703,0.999875,0.999663,0.999875,0.999769,0.001299,10.819448,10092.0,1.000000
1,species,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.999505,0.999792,0.999334,0.999792,0.999563,0.001298,11.597037,10092.0,1.000000
2,species,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.999405,0.999597,0.999236,0.999597,0.999416,0.003117,11.299483,10092.0,0.999901
3,species,test,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+ConvNeXtTiny_ft_...,EfficientNetB0+ConvNeXtTiny+EfficientNetB1,0.999802,0.999917,0.999761,0.999917,0.999839,0.001284,33.715969,10092.0,1.000000
4,species,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.676141,0.627862,0.652486,0.627862,0.605973,2.292471,14.080059,2322.0,0.835056
5,species,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.739879,0.742131,0.682576,0.742131,0.705540,1.480664,15.390665,2322.0,0.892334
6,species,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.714040,0.675597,0.664698,0.675597,0.651511,1.770179,14.408032,2322.0,0.867786
7,species,ood,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+ConvNeXtTiny_ft_...,EfficientNetB0+ConvNeXtTiny+EfficientNetB1,0.756245,0.720750,0.721270,0.720750,0.708147,1.043444,43.878755,2322.0,0.890181


Évaluation species sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/species_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,species,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.999703,0.999875,0.999663,0.999875,0.999769,0.001299,10.819448,10092.0,1.000000
1,species,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.999505,0.999792,0.999334,0.999792,0.999563,0.001298,11.597037,10092.0,1.000000
2,species,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.999405,0.999597,0.999236,0.999597,0.999416,0.003117,11.299483,10092.0,0.999901
3,species,test,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+ConvNeXtTiny_ft_...,EfficientNetB0+ConvNeXtTiny+EfficientNetB1,0.999802,0.999917,0.999761,0.999917,0.999839,0.001284,33.715969,10092.0,1.000000
4,species,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.676141,0.627862,0.652486,0.627862,0.605973,2.292471,14.080059,2322.0,0.835056
5,species,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.739879,0.742131,0.682576,0.742131,0.705540,1.480664,15.390665,2322.0,0.892334
6,species,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.714040,0.675597,0.664698,0.675597,0.651511,1.770179,14.408032,2322.0,0.867786
7,species,ood,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+ConvNeXtTiny_ft_...,EfficientNetB0+ConvNeXtTiny+EfficientNetB1,0.756245,0.720750,0.721270,0.720750,0.708147,1.043444,43.878755,2322.0,0.890181


### Évaluation tomate

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [11]:
evaluate_task_ensemble("tomato")


tomato: 3 modèle(s), 10 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/tomato_ensemble_evaluation.csv
  - test: 3438 images


2026-04-17 19:12:54.045642: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1522', 12 bytes spill stores, 12 bytes spill loads

2026-04-17 19:12:54.107223: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1522', 248 bytes spill stores, 248 bytes spill loads

2026-04-17 19:12:54.244228: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1522', 16 bytes spill stores, 16 bytes spill loads

2026-04-17 19:12:54.313721: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1522', 16 bytes spill stores, 16 bytes spill loads

2026-04-17 19:12:54.396293: I external

    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 903 images


2026-04-17 19:14:35.532539: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1', 184 bytes spill stores, 184 bytes spill loads

2026-04-17 19:14:35.583293: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 484 bytes spill stores, 452 bytes spill loads

2026-04-17 19:14:35.687326: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 448 bytes spill stores, 416 bytes spill loads

2026-04-17 19:14:35.733777: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_6', 116 bytes spill stores, 116 bytes spill loads

2026-04-17 19:14:35.809385: I external/local_xla

    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,tomato,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.989820,0.989717,0.989946,0.989717,0.989785,0.025737,12.332481,3438.0,0.999418
1,tomato,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.988365,0.988282,0.988272,0.988282,0.988261,0.044046,12.487553,3438.0,0.998837
2,tomato,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.985166,0.985050,0.984823,0.985050,0.984912,0.050371,11.938418,3438.0,0.998837
3,tomato,test,ensemble_soft_vote,ConvNeXtTiny_ft_l50_lr1e_05+MobileNetV3Large_f...,ConvNeXtTiny+MobileNetV3Large+EfficientNetB0,0.993892,0.993885,0.993783,0.993885,0.993827,0.028634,36.758451,3438.0,0.999418
4,tomato,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.265781,0.238402,0.244988,0.214562,0.175945,4.646029,21.196888,903.0,0.444075
5,tomato,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.225914,0.208062,0.247983,0.187256,0.137239,7.300358,17.290452,903.0,0.411960
6,tomato,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.252492,0.207931,0.299011,0.187138,0.156663,5.086629,18.783826,903.0,0.477298
7,tomato,ood,ensemble_soft_vote,ConvNeXtTiny_ft_l50_lr1e_05+MobileNetV3Large_f...,ConvNeXtTiny+MobileNetV3Large+EfficientNetB0,0.252492,0.228409,0.264484,0.205568,0.162487,3.965015,57.271166,903.0,0.445183


Évaluation tomato sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/tomato_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,tomato,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.989820,0.989717,0.989946,0.989717,0.989785,0.025737,12.332481,3438.0,0.999418
1,tomato,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.988365,0.988282,0.988272,0.988282,0.988261,0.044046,12.487553,3438.0,0.998837
2,tomato,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.985166,0.985050,0.984823,0.985050,0.984912,0.050371,11.938418,3438.0,0.998837
3,tomato,test,ensemble_soft_vote,ConvNeXtTiny_ft_l50_lr1e_05+MobileNetV3Large_f...,ConvNeXtTiny+MobileNetV3Large+EfficientNetB0,0.993892,0.993885,0.993783,0.993885,0.993827,0.028634,36.758451,3438.0,0.999418
4,tomato,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.265781,0.238402,0.244988,0.214562,0.175945,4.646029,21.196888,903.0,0.444075
5,tomato,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.225914,0.208062,0.247983,0.187256,0.137239,7.300358,17.290452,903.0,0.411960
6,tomato,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.252492,0.207931,0.299011,0.187138,0.156663,5.086629,18.783826,903.0,0.477298
7,tomato,ood,ensemble_soft_vote,ConvNeXtTiny_ft_l50_lr1e_05+MobileNetV3Large_f...,ConvNeXtTiny+MobileNetV3Large+EfficientNetB0,0.252492,0.228409,0.264484,0.205568,0.162487,3.965015,57.271166,903.0,0.445183


### Évaluation pommier

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [12]:
evaluate_task_ensemble("apple")


apple: 3 modèle(s), 4 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/apple_ensemble_evaluation.csv
  - test: 1457 images


2026-04-17 19:24:08.389044: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 280 bytes spill stores, 280 bytes spill loads

2026-04-17 19:24:08.410309: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 64 bytes spill stores, 64 bytes spill loads

2026-04-17 19:24:08.552946: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 64 bytes spill stores, 64 bytes spill loads

2026-04-17 19:24:08.639088: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 716 bytes spill stores, 720 bytes spill loads

2026-04-17 19:24:08.910660: I extern

    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 287 images


2026-04-17 19:24:40.790985: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
2026-04-17 19:24:57.378432: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 64 bytes spill stores, 64 bytes spill loads

2026-04-17 19:24:57.393941: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Register

    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,apple,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,1.000000,1.000000,1.000000,1.000000,1.000000,0.002130,15.226462,1457.0,1.000000
1,apple,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.999314,0.999337,0.999340,0.999337,0.999338,0.003081,13.690637,1457.0,1.000000
2,apple,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.998627,0.998675,0.998668,0.998675,0.998671,0.005743,15.257289,1457.0,1.000000
3,apple,test,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+MobileNetV3Large...,EfficientNetB0+MobileNetV3Large+MobileNetV3Small,0.999314,0.999337,0.999340,0.999337,0.999338,0.002939,44.174388,1457.0,1.000000
4,apple,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.376307,0.387191,0.459439,0.290393,0.331464,3.348272,29.675950,287.0,0.658537
5,apple,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.337979,0.343660,0.541166,0.257745,0.297298,5.671213,25.237746,287.0,0.616725
6,apple,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.376307,0.381531,0.522574,0.286149,0.311666,4.584566,31.767274,287.0,0.648084
7,apple,ood,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+MobileNetV3Large...,EfficientNetB0+MobileNetV3Large+MobileNetV3Small,0.365854,0.376568,0.494176,0.282426,0.314578,2.840474,86.680970,287.0,0.623693


Évaluation apple sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/apple_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,apple,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,1.000000,1.000000,1.000000,1.000000,1.000000,0.002130,15.226462,1457.0,1.000000
1,apple,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.999314,0.999337,0.999340,0.999337,0.999338,0.003081,13.690637,1457.0,1.000000
2,apple,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.998627,0.998675,0.998668,0.998675,0.998671,0.005743,15.257289,1457.0,1.000000
3,apple,test,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+MobileNetV3Large...,EfficientNetB0+MobileNetV3Large+MobileNetV3Small,0.999314,0.999337,0.999340,0.999337,0.999338,0.002939,44.174388,1457.0,1.000000
4,apple,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.376307,0.387191,0.459439,0.290393,0.331464,3.348272,29.675950,287.0,0.658537
5,apple,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.337979,0.343660,0.541166,0.257745,0.297298,5.671213,25.237746,287.0,0.616725
6,apple,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.376307,0.381531,0.522574,0.286149,0.311666,4.584566,31.767274,287.0,0.648084
7,apple,ood,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+MobileNetV3Large...,EfficientNetB0+MobileNetV3Large+MobileNetV3Small,0.365854,0.376568,0.494176,0.282426,0.314578,2.840474,86.680970,287.0,0.623693


### Évaluation vigne

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [13]:
evaluate_task_ensemble("grape")


grape: 3 modèle(s), 4 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/grape_ensemble_evaluation.csv
  - test: 1355 images
    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 154 images


/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,grape,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,1.000000,1.000000,1.000000,1.000000,1.000000,0.001113,17.065179,1355.0,1.000000
1,grape,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.999262,0.999306,0.999296,0.999306,0.999300,0.002665,14.740673,1355.0,1.000000
2,grape,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.998524,0.998611,0.998462,0.998611,0.998532,0.003390,14.307409,1355.0,1.000000
3,grape,test,ensemble_soft_vote,EfficientNetB1_ft_l50_lr1e_05+EfficientNetB0_f...,EfficientNetB1+EfficientNetB0+ConvNeXtTiny,1.000000,1.000000,1.000000,1.000000,1.000000,0.001888,46.113261,1355.0,1.000000
4,grape,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.525974,0.524135,0.398214,0.262068,0.314364,2.057148,58.785658,154.0,0.720779
5,grape,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.597403,0.594430,0.390244,0.297215,0.329085,1.911596,48.790670,154.0,0.811688
6,grape,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.571429,0.577215,0.404378,0.288608,0.315952,1.989753,42.373206,154.0,0.733766
7,grape,ood,ensemble_soft_vote,EfficientNetB1_ft_l50_lr1e_05+EfficientNetB0_f...,EfficientNetB1+EfficientNetB0+ConvNeXtTiny,0.629870,0.629114,0.411765,0.314557,0.355442,1.098447,149.949534,154.0,0.850649


Évaluation grape sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/grape_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,grape,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,1.000000,1.000000,1.000000,1.000000,1.000000,0.001113,17.065179,1355.0,1.000000
1,grape,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.999262,0.999306,0.999296,0.999306,0.999300,0.002665,14.740673,1355.0,1.000000
2,grape,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.998524,0.998611,0.998462,0.998611,0.998532,0.003390,14.307409,1355.0,1.000000
3,grape,test,ensemble_soft_vote,EfficientNetB1_ft_l50_lr1e_05+EfficientNetB0_f...,EfficientNetB1+EfficientNetB0+ConvNeXtTiny,1.000000,1.000000,1.000000,1.000000,1.000000,0.001888,46.113261,1355.0,1.000000
4,grape,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.525974,0.524135,0.398214,0.262068,0.314364,2.057148,58.785658,154.0,0.720779
5,grape,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.597403,0.594430,0.390244,0.297215,0.329085,1.911596,48.790670,154.0,0.811688
6,grape,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.571429,0.577215,0.404378,0.288608,0.315952,1.989753,42.373206,154.0,0.733766
7,grape,ood,ensemble_soft_vote,EfficientNetB1_ft_l50_lr1e_05+EfficientNetB0_f...,EfficientNetB1+EfficientNetB0+ConvNeXtTiny,0.629870,0.629114,0.411765,0.314557,0.355442,1.098447,149.949534,154.0,0.850649


### Évaluation maïs

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [14]:
evaluate_task_ensemble("corn")


corn: 3 modèle(s), 4 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/corn_ensemble_evaluation.csv
  - test: 1370 images
    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 378 images


/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,corn,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.991971,0.991183,0.992333,0.991183,0.991666,0.022140,13.853286,1370.0,1.000000
1,corn,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.991241,0.990705,0.991170,0.990705,0.990920,0.025397,16.090291,1370.0,1.000000
2,corn,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.990511,0.990005,0.990439,0.990005,0.990207,0.028085,15.056053,1370.0,1.000000
3,corn,test,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+EfficientNetB0,0.991971,0.991294,0.992269,0.991294,0.991715,0.020067,44.999631,1370.0,1.000000
4,corn,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.309524,0.375049,0.371082,0.281287,0.223210,5.692658,21.811084,378.0,0.542328
5,corn,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.267196,0.312862,0.379341,0.234647,0.227760,4.038927,31.684614,378.0,0.560847
6,corn,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.314815,0.380158,0.381128,0.285118,0.250792,3.486415,25.824235,378.0,0.582011
7,corn,ood,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+EfficientNetB0,0.275132,0.332040,0.371109,0.249030,0.207527,3.248463,79.319933,378.0,0.521164


Évaluation corn sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/corn_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,corn,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.991971,0.991183,0.992333,0.991183,0.991666,0.022140,13.853286,1370.0,1.000000
1,corn,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.991241,0.990705,0.991170,0.990705,0.990920,0.025397,16.090291,1370.0,1.000000
2,corn,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.990511,0.990005,0.990439,0.990005,0.990207,0.028085,15.056053,1370.0,1.000000
3,corn,test,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+EfficientNetB0,0.991971,0.991294,0.992269,0.991294,0.991715,0.020067,44.999631,1370.0,1.000000
4,corn,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.309524,0.375049,0.371082,0.281287,0.223210,5.692658,21.811084,378.0,0.542328
5,corn,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.267196,0.312862,0.379341,0.234647,0.227760,4.038927,31.684614,378.0,0.560847
6,corn,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.314815,0.380158,0.381128,0.285118,0.250792,3.486415,25.824235,378.0,0.582011
7,corn,ood,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+EfficientNetB0,0.275132,0.332040,0.371109,0.249030,0.207527,3.248463,79.319933,378.0,0.521164


### Évaluation pomme de terre

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [15]:
evaluate_task_ensemble("potato")


potato: 3 modèle(s), 3 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/potato_ensemble_evaluation.csv
  - test: 1068 images


2026-04-17 19:39:29.508229: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 379 images


/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,potato,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,1.000000,1.000000,1.000000,1.000000,1.000000,0.000751,13.497565,1068.0,1.000000
1,potato,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.999064,0.999082,0.999028,0.999082,0.999054,0.003036,15.032123,1068.0,1.000000
2,potato,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.999064,0.999025,0.999084,0.999025,0.999053,0.003878,12.418770,1068.0,1.000000
3,potato,test,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+MobileNetV3Small,1.000000,1.000000,1.000000,1.000000,1.000000,0.002336,40.948458,1068.0,1.000000
4,potato,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.633245,0.616453,0.430066,0.410969,0.411558,2.673509,24.011187,379.0,0.928760
5,potato,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.559367,0.558508,0.382187,0.372338,0.376690,2.155437,32.545977,379.0,0.912929
6,potato,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.535620,0.543635,0.367655,0.362423,0.359187,2.476068,21.470871,379.0,0.968338
7,potato,ood,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+MobileNetV3Small,0.585752,0.580466,0.391458,0.386977,0.389128,1.554412,78.028034,379.0,0.944591


Évaluation potato sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/potato_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy
0,potato,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,1.000000,1.000000,1.000000,1.000000,1.000000,0.000751,13.497565,1068.0,1.000000
1,potato,test,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.999064,0.999082,0.999028,0.999082,0.999054,0.003036,15.032123,1068.0,1.000000
2,potato,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.999064,0.999025,0.999084,0.999025,0.999053,0.003878,12.418770,1068.0,1.000000
3,potato,test,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+MobileNetV3Small,1.000000,1.000000,1.000000,1.000000,1.000000,0.002336,40.948458,1068.0,1.000000
4,potato,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.633245,0.616453,0.430066,0.410969,0.411558,2.673509,24.011187,379.0,0.928760
5,potato,ood,single,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,0.559367,0.558508,0.382187,0.372338,0.376690,2.155437,32.545977,379.0,0.912929
6,potato,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.535620,0.543635,0.367655,0.362423,0.359187,2.476068,21.470871,379.0,0.968338
7,potato,ood,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,MobileNetV3Large+EfficientNetB1+MobileNetV3Small,0.585752,0.580466,0.391458,0.386977,0.389128,1.554412,78.028034,379.0,0.944591


### Évaluation poivron

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [16]:
evaluate_task_ensemble("pepper")


pepper: 3 modèle(s), 2 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/pepper_ensemble_evaluation.csv
  - test: 730 images


2026-04-17 19:41:57.318022: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 64 bytes spill stores, 64 bytes spill loads

2026-04-17 19:41:57.327214: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 280 bytes spill stores, 280 bytes spill loads

2026-04-17 19:41:57.643697: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 64 bytes spill stores, 64 bytes spill loads

2026-04-17 19:41:57.664810: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1307', 4352 bytes spill stores, 4456 bytes spill loads

2026-04-17 19:41:57.738955: I exte

    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 125 images
    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count
0,pepper,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,1.00000,1.000000,1.000000,1.000000,1.000000,0.002800,18.051468,730.0
1,pepper,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,1.00000,1.000000,1.000000,1.000000,1.000000,0.003277,18.811989,730.0
2,pepper,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.99726,0.997207,0.997326,0.997207,0.997259,0.007907,17.432187,730.0
3,pepper,test,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+MobileNetV3Sma...,MobileNetV3Large+MobileNetV3Small+ConvNeXtTiny,1.00000,1.000000,1.000000,1.000000,1.000000,0.003818,54.295644,730.0
4,pepper,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.80800,0.737808,0.822067,0.737808,0.757595,0.934676,45.352895,125.0
5,pepper,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.73600,0.660069,0.710884,0.660069,0.669709,1.158952,41.490377,125.0
6,pepper,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.80000,0.843517,0.807051,0.843517,0.795765,1.033663,47.012933,125.0
7,pepper,ood,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+MobileNetV3Sma...,MobileNetV3Large+MobileNetV3Small+ConvNeXtTiny,0.81600,0.773236,0.802381,0.773236,0.784175,0.364324,133.856205,125.0


Évaluation pepper sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/pepper_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count
0,pepper,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,1.00000,1.000000,1.000000,1.000000,1.000000,0.002800,18.051468,730.0
1,pepper,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,1.00000,1.000000,1.000000,1.000000,1.000000,0.003277,18.811989,730.0
2,pepper,test,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.99726,0.997207,0.997326,0.997207,0.997259,0.007907,17.432187,730.0
3,pepper,test,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+MobileNetV3Sma...,MobileNetV3Large+MobileNetV3Small+ConvNeXtTiny,1.00000,1.000000,1.000000,1.000000,1.000000,0.003818,54.295644,730.0
4,pepper,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.80800,0.737808,0.822067,0.737808,0.757595,0.934676,45.352895,125.0
5,pepper,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.73600,0.660069,0.710884,0.660069,0.669709,1.158952,41.490377,125.0
6,pepper,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.80000,0.843517,0.807051,0.843517,0.795765,1.033663,47.012933,125.0
7,pepper,ood,ensemble_soft_vote,MobileNetV3Large_ft_l50_lr1e_05+MobileNetV3Sma...,MobileNetV3Large+MobileNetV3Small+ConvNeXtTiny,0.81600,0.773236,0.802381,0.773236,0.784175,0.364324,133.856205,125.0


### Évaluation fraisier

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [17]:
evaluate_task_ensemble("strawberry")


strawberry: 3 modèle(s), 2 classe(s)
Sauvegarde: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/strawberry_ensemble_evaluation.csv
  - test: 674 images
    sauvegarde intermédiaire: 4 ligne(s)
  - ood: 96 images


/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/thomashebert99/.pyenv/versions/plant-disease-detection/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


    sauvegarde intermédiaire: 8 ligne(s)


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count
0,strawberry,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,1.000000,1.000000,1.0,1.000000,1.000000,0.000316,17.603483,674.0
1,strawberry,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,1.000000,1.000000,1.0,1.000000,1.000000,0.000100,19.214360,674.0
2,strawberry,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,1.000000,1.000000,1.0,1.000000,1.000000,0.000763,16.930541,674.0
3,strawberry,test,ensemble_soft_vote,MobileNetV3Small_ft_l50_lr1e_05+EfficientNetB0...,MobileNetV3Small+EfficientNetB0+MobileNetV3Large,1.000000,1.000000,1.0,1.000000,1.000000,0.000364,53.748385,674.0
4,strawberry,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.697917,0.697917,0.5,0.348958,0.411043,1.719645,31.548738,96.0
5,strawberry,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.437500,0.437500,0.5,0.218750,0.304348,3.110696,41.258828,96.0
6,strawberry,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.614583,0.614583,0.5,0.307292,0.380645,2.342383,35.347559,96.0
7,strawberry,ood,ensemble_soft_vote,MobileNetV3Small_ft_l50_lr1e_05+EfficientNetB0...,MobileNetV3Small+EfficientNetB0+MobileNetV3Large,0.604167,0.604167,0.5,0.302083,0.376623,1.767215,108.155125,96.0


Évaluation strawberry sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/evaluations/strawberry_ensemble_evaluation.csv


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count
0,strawberry,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,1.000000,1.000000,1.0,1.000000,1.000000,0.000316,17.603483,674.0
1,strawberry,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,1.000000,1.000000,1.0,1.000000,1.000000,0.000100,19.214360,674.0
2,strawberry,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,1.000000,1.000000,1.0,1.000000,1.000000,0.000763,16.930541,674.0
3,strawberry,test,ensemble_soft_vote,MobileNetV3Small_ft_l50_lr1e_05+EfficientNetB0...,MobileNetV3Small+EfficientNetB0+MobileNetV3Large,1.000000,1.000000,1.0,1.000000,1.000000,0.000364,53.748385,674.0
4,strawberry,ood,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.697917,0.697917,0.5,0.348958,0.411043,1.719645,31.548738,96.0
5,strawberry,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.437500,0.437500,0.5,0.218750,0.304348,3.110696,41.258828,96.0
6,strawberry,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.614583,0.614583,0.5,0.307292,0.380645,2.342383,35.347559,96.0
7,strawberry,ood,ensemble_soft_vote,MobileNetV3Small_ft_l50_lr1e_05+EfficientNetB0...,MobileNetV3Small+EfficientNetB0+MobileNetV3Large,0.604167,0.604167,0.5,0.302083,0.376623,1.767215,108.155125,96.0


### Recharger toutes les évaluations sauvegardées

Exécuter cette cellule avant la synthèse si le kernel a été redémarré entre deux tâches.

In [18]:
evaluation = load_saved_evaluations()
if evaluation.empty:
    print("Aucune évaluation sauvegardée.")
else:
    display(evaluation)
    print(f"Évaluation globale sauvegardée dans: {ENSEMBLE_OUTPUT_DIR / 'ensemble_evaluation.csv'}")


,task,dataset,model_type,run_name,architecture,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,log_loss,ms_per_image,image_count,top2_accuracy,evaluation_file
0,apple,test,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,1.000000,1.000000,1.000000,1.000000,1.000000,0.002130,15.226462,1457.0,1.000000,/home/thomashebert99/code/thomashebert99/Plant...
1,apple,test,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.999314,0.999337,0.999340,0.999337,0.999338,0.003081,13.690637,1457.0,1.000000,/home/thomashebert99/code/thomashebert99/Plant...
2,apple,test,single,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,0.998627,0.998675,0.998668,0.998675,0.998671,0.005743,15.257289,1457.0,1.000000,/home/thomashebert99/code/thomashebert99/Plant...
3,apple,test,ensemble_soft_vote,EfficientNetB0_ft_l50_lr1e_05+MobileNetV3Large...,EfficientNetB0+MobileNetV3Large+MobileNetV3Small,0.999314,0.999337,0.999340,0.999337,0.999338,0.002939,44.174388,1457.0,1.000000,/home/thomashebert99/code/thomashebert99/Plant...
4,apple,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.376307,0.387191,0.459439,0.290393,0.331464,3.348272,29.675950,287.0,0.658537,/home/thomashebert99/code/thomashebert99/Plant...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,tomato,test,ensemble_soft_vote,ConvNeXtTiny_ft_l50_lr1e_05+MobileNetV3Large_f...,ConvNeXtTiny+MobileNetV3Large+EfficientNetB0,0.993892,0.993885,0.993783,0.993885,0.993827,0.028634,36.758451,3438.0,0.999418,/home/thomashebert99/code/thomashebert99/Plant...
60,tomato,ood,single,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,0.265781,0.238402,0.244988,0.214562,0.175945,4.646029,21.196888,903.0,0.444075,/home/thomashebert99/code/thomashebert99/Plant...
61,tomato,ood,single,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,0.225914,0.208062,0.247983,0.187256,0.137239,7.300358,17.290452,903.0,0.411960,/home/thomashebert99/code/thomashebert99/Plant...
62,tomato,ood,single,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,0.252492,0.207931,0.299011,0.187138,0.156663,5.086629,18.783826,903.0,0.477298,/home/thomashebert99/code/thomashebert99/Plant...


Évaluation globale sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/ensemble_evaluation.csv


## Synthèse du gain

On compare l'ensemble au meilleur modèle seul observé dans la même évaluation.

Cette mesure sert à documenter l'intérêt réel du vote doux :

- gain positif : l'ensemble améliore la tâche ;
- gain proche de zéro : l'ensemble stabilise la décision sans perte notable ;
- gain négatif : le rapport doit le signaler honnêtement.

Pour garder une architecture claire et cohérente avec le projet, la configuration finale conserve trois modèles par tâche. Le vote doux reste donc la stratégie de production, et les gains sont utilisés comme analyse critique plutôt que comme règle de fallback automatique.


In [19]:
def gain_summary(evaluation: pd.DataFrame) -> pd.DataFrame:
    if evaluation.empty:
        return evaluation

    rows = []
    for (task, dataset), group in evaluation.groupby(["task", "dataset"]):
        singles = group[group["model_type"] == "single"]
        ensembles = group[group["model_type"] == "ensemble_soft_vote"]
        if singles.empty or ensembles.empty:
            continue

        best_single = singles.sort_values("f1_macro", ascending=False).iloc[0]
        ensemble = ensembles.iloc[0]
        rows.append(
            {
                "task": task,
                "dataset": dataset,
                "best_single": best_single["run_name"],
                "ensemble_models": ensemble["run_name"],
                "best_single_f1_macro": best_single["f1_macro"],
                "ensemble_f1_macro": ensemble["f1_macro"],
                "gain_f1_macro": ensemble["f1_macro"] - best_single["f1_macro"],
                "best_single_accuracy": best_single["accuracy"],
                "ensemble_accuracy": ensemble["accuracy"],
                "gain_accuracy": ensemble["accuracy"] - best_single["accuracy"],
                "best_single_log_loss": best_single["log_loss"],
                "ensemble_log_loss": ensemble["log_loss"],
                "gain_log_loss": best_single["log_loss"] - ensemble["log_loss"],
            }
        )
    return pd.DataFrame(rows)


def build_final_decisions(selected: pd.DataFrame, gains: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task in EXPECTED_TASKS:
        task_selection = selected[selected["task"] == task] if not selected.empty else pd.DataFrame()
        if task_selection.empty:
            continue

        task_gains = gains[(gains["task"] == task) & (gains["dataset"] == "test")] if not gains.empty else pd.DataFrame()
        gain_f1 = float(task_gains.iloc[0]["gain_f1_macro"]) if not task_gains.empty else None
        gain_log_loss = float(task_gains.iloc[0]["gain_log_loss"]) if not task_gains.empty else None
        rows.append(
            {
                "task": task,
                "final_strategy": "soft_vote_mean_probabilities",
                "model_count": int(len(task_selection)),
                "gain_f1_macro_test": gain_f1,
                "gain_log_loss_test": gain_log_loss,
                "reason": "Architecture d'ensemble conservée : trois modèles sélectionnés votent pour chaque tâche.",
            }
        )
    return pd.DataFrame(rows)


gains = gain_summary(evaluation)
gains_path = ENSEMBLE_OUTPUT_DIR / "ensemble_gain_summary.csv"
if not gains.empty:
    gains.to_csv(gains_path, index=False)
    display(gains)
    print(f"Synthèse sauvegardée dans: {gains_path}")
else:
    print("Pas encore de synthèse de gain.")

final_models = selected_models.copy()
if not final_models.empty:
    final_models["final_strategy"] = "soft_vote_mean_probabilities"
    final_models["final_decision_reason"] = "Architecture d'ensemble conservée : trois modèles sélectionnés votent pour chaque tâche."

final_decisions = build_final_decisions(selected_models, gains)
final_selection_path = ENSEMBLE_OUTPUT_DIR / "final_selection_summary.csv"
final_decisions_path = ENSEMBLE_OUTPUT_DIR / "final_decisions.csv"

if not final_models.empty:
    final_models.to_csv(final_selection_path, index=False)
    final_decisions.to_csv(final_decisions_path, index=False)
    display(final_decisions)
    display(final_models[[column for column in [
        "task", "selected_rank", "final_strategy", "run_name", "architecture",
        "architecture_family", "eval_test_f1_macro", "final_decision_reason",
    ] if column in final_models.columns]])
    print(f"Sélection finale sauvegardée dans: {final_selection_path}")
    print(f"Décisions finales sauvegardées dans: {final_decisions_path}")
else:
    print("Pas encore de sélection finale.")


,task,dataset,best_single,ensemble_models,best_single_f1_macro,ensemble_f1_macro,gain_f1_macro,best_single_accuracy,ensemble_accuracy,gain_accuracy,best_single_log_loss,ensemble_log_loss,gain_log_loss
0,apple,ood,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0_ft_l50_lr1e_05+MobileNetV3Large...,0.331464,0.314578,-0.016886,0.376307,0.365854,-0.010453,3.348272,2.840474,0.507798
1,apple,test,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0_ft_l50_lr1e_05+MobileNetV3Large...,1.000000,0.999338,-0.000662,1.000000,0.999314,-0.000686,0.002130,0.002939,-0.000809
2,corn,ood,EfficientNetB0_ft_l50_lr1e_05,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,0.250792,0.207527,-0.043265,0.314815,0.275132,-0.039683,3.486415,3.248463,0.237952
3,corn,test,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,0.991666,0.991715,0.000049,0.991971,0.991971,0.000000,0.022140,0.020067,0.002073
4,grape,ood,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB1_ft_l50_lr1e_05+EfficientNetB0_f...,0.329085,0.355442,0.026357,0.597403,0.629870,0.032468,1.911596,1.098447,0.813149
5,grape,test,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1_ft_l50_lr1e_05+EfficientNetB0_f...,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.001113,0.001888,-0.000775
6,pepper,ood,ConvNeXtTiny_ft_l50_lr1e_05,MobileNetV3Large_ft_l50_lr1e_05+MobileNetV3Sma...,0.795765,0.784175,-0.011590,0.800000,0.816000,0.016000,1.033663,0.364324,0.669339
7,pepper,test,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large_ft_l50_lr1e_05+MobileNetV3Sma...,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.002800,0.003818,-0.001018
8,potato,ood,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,0.411558,0.389128,-0.022430,0.633245,0.585752,-0.047493,2.673509,1.554412,1.119097
9,potato,test,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large_ft_l50_lr1e_05+EfficientNetB1...,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000751,0.002336,-0.001585


Synthèse sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/ensemble_gain_summary.csv


,task,final_strategy,model_count,gain_f1_macro_test,gain_log_loss_test,reason
0,species,soft_vote_mean_probabilities,3,0.000070,0.000014,Architecture d'ensemble conservée : trois modè...
1,tomato,soft_vote_mean_probabilities,3,0.004042,-0.002896,Architecture d'ensemble conservée : trois modè...
2,apple,soft_vote_mean_probabilities,3,-0.000662,-0.000809,Architecture d'ensemble conservée : trois modè...
3,grape,soft_vote_mean_probabilities,3,0.000000,-0.000775,Architecture d'ensemble conservée : trois modè...
4,corn,soft_vote_mean_probabilities,3,0.000049,0.002073,Architecture d'ensemble conservée : trois modè...
5,potato,soft_vote_mean_probabilities,3,0.000000,-0.001585,Architecture d'ensemble conservée : trois modè...
6,pepper,soft_vote_mean_probabilities,3,0.000000,-0.001018,Architecture d'ensemble conservée : trois modè...
7,strawberry,soft_vote_mean_probabilities,3,0.000000,-0.000048,Architecture d'ensemble conservée : trois modè...


,task,selected_rank,final_strategy,run_name,architecture,architecture_family,eval_test_f1_macro,final_decision_reason
0,species,1,soft_vote_mean_probabilities,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.999769,Architecture d'ensemble conservée : trois modè...
1,species,2,soft_vote_mean_probabilities,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,ConvNeXt,0.999563,Architecture d'ensemble conservée : trois modè...
2,species,3,soft_vote_mean_probabilities,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,0.999416,Architecture d'ensemble conservée : trois modè...
3,tomato,1,soft_vote_mean_probabilities,ConvNeXtTiny_ft_l50_lr1e_05,ConvNeXtTiny,ConvNeXt,0.989785,Architecture d'ensemble conservée : trois modè...
4,tomato,2,soft_vote_mean_probabilities,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,0.988261,Architecture d'ensemble conservée : trois modè...
5,tomato,3,soft_vote_mean_probabilities,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,0.984912,Architecture d'ensemble conservée : trois modè...
6,apple,1,soft_vote_mean_probabilities,EfficientNetB0_ft_l50_lr1e_05,EfficientNetB0,EfficientNet,1.000000,Architecture d'ensemble conservée : trois modè...
7,apple,2,soft_vote_mean_probabilities,MobileNetV3Large_ft_l50_lr1e_05,MobileNetV3Large,MobileNet,0.999338,Architecture d'ensemble conservée : trois modè...
8,apple,3,soft_vote_mean_probabilities,MobileNetV3Small_ft_l50_lr1e_05,MobileNetV3Small,MobileNet,0.998671,Architecture d'ensemble conservée : trois modè...
9,grape,1,soft_vote_mean_probabilities,EfficientNetB1_ft_l50_lr1e_05,EfficientNetB1,EfficientNet,1.000000,Architecture d'ensemble conservée : trois modè...


Sélection finale sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/final_selection_summary.csv
Décisions finales sauvegardées dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble/final_decisions.csv


## Sauvegarder la configuration finale

La configuration est volontairement plus riche qu'une simple liste de noms. Elle contient les chemins de checkpoints, l'ordre des classes, les métriques de sélection et la stratégie finale retenue.

C'est ce fichier que l'API lit ensuite pour charger les bons modèles. Chaque tâche conserve trois modèles et utilise le vote doux, ce qui garde l'architecture de production simple à expliquer et cohérente avec le projet.

Comme `REQUIRE_COMPLETE_RESULTS=True`, le fichier final est écrit dans `models/ensemble_config.json` seulement quand toutes les tâches attendues disposent de trois modèles sélectionnés.


In [20]:
def row_float(row: pd.Series, key: str) -> float | None:
    value = row.get(key)
    if pd.isna(value):
        return None
    return float(value)


def build_ensemble_config(final_selected: pd.DataFrame, gains: pd.DataFrame, decisions: pd.DataFrame) -> dict[str, object]:
    tasks = {}
    for task, task_selection in final_selected.groupby("task") if not final_selected.empty else []:
        class_names = json.loads(task_selection.iloc[0]["class_names"])
        task_gains = gains[gains["task"] == task] if not gains.empty else pd.DataFrame()
        task_decision = decisions[decisions["task"] == task] if not decisions.empty else pd.DataFrame()
        decision_payload = task_decision.iloc[0].to_dict() if not task_decision.empty else {}

        models = []
        for _, row in task_selection.sort_values("selected_rank").iterrows():
            checkpoint_path = resolve_checkpoint_path(row["checkpoint_path"])
            models.append(
                {
                    "run_name": row["run_name"],
                    "architecture": row["architecture"],
                    "architecture_family": row.get("architecture_family"),
                    "checkpoint_path": str(checkpoint_path),
                    "selected_rank": int(row["selected_rank"]),
                    "ranking_score": row_float(row, "ranking_score"),
                    "eval_test_f1_macro": row_float(row, "eval_test_f1_macro"),
                    "eval_ood_f1_macro": row_float(row, "eval_ood_f1_macro"),
                }
            )

        tasks[task] = {
            "task_type": task_selection.iloc[0]["task_type"],
            "display_name": task_selection.iloc[0].get("display_name", task),
            "strategy": "soft_vote_mean_probabilities",
            "selection_policy": "top3_max2_family",
            "decision_reason": decision_payload.get("reason"),
            "image_size": IMAGE_SIZE,
            "class_names": class_names,
            "models": models,
            "gains": task_gains.to_dict(orient="records"),
        }

    complete_tasks = sorted(task for task, payload in tasks.items() if len(payload["models"]) >= ENSEMBLE_SIZE)
    missing_tasks = sorted(set(EXPECTED_TASKS) - set(complete_tasks))
    return {
        "version": 1,
        "created_by": "notebooks/05_ensemble_selection.ipynb",
        "selection_source": "local CSV files from notebooks 03 and 04",
        "selection_policy": "top3_max2_family",
        "complete": len(missing_tasks) == 0,
        "complete_tasks": complete_tasks,
        "missing_tasks": missing_tasks,
        "tasks": tasks,
    }


ensemble_config = build_ensemble_config(final_models, gains, final_decisions)
if REQUIRE_COMPLETE_RESULTS and not ensemble_config["complete"]:
    raise RuntimeError(f"Configuration incomplète, tâches manquantes: {ensemble_config['missing_tasks']}")

config_path = MODEL_ROOT / "ensemble_config.json" if ensemble_config["complete"] else ENSEMBLE_OUTPUT_DIR / "ensemble_config_partial.json"
config_path.write_text(json.dumps(ensemble_config, indent=2), encoding="utf-8")

print(f"Configuration sauvegardée dans: {config_path}")
print(f"Configuration complète: {ensemble_config['complete']}")
print(f"Tâches prêtes: {ensemble_config['complete_tasks']}")
print(f"Tâches manquantes: {ensemble_config['missing_tasks']}")


Configuration sauvegardée dans: /home/thomashebert99/code/thomashebert99/Plant-disease-detection/models/ensemble_config.json
Configuration complète: True
Tâches prêtes: ['apple', 'corn', 'grape', 'pepper', 'potato', 'species', 'strawberry', 'tomato']
Tâches manquantes: []


## Conclusion à reporter

À la fin de l'exécution complète, reprendre quatre éléments dans le rapport :

1. Le tableau de comparaison des stratégies (`selection_strategy_comparison.csv`).
2. Le tableau de sélection candidate (`selection_summary.csv`).
3. Le tableau de gain de l'ensemble (`ensemble_gain_summary.csv`).
4. Le tableau de décision finale (`final_decisions.csv`), qui rappelle que l'architecture de production conserve le vote doux à trois modèles.

La justification principale doit rester simple : le projet privilégie une architecture d'ensemble homogène, avec une sélection mesurée sur les résultats et une diversité raisonnable, sans forcer une architecture moins bonne uniquement pour remplir une contrainte artificielle.
